# COCO Lab — Detección y Segmentación de Instancias con YOLO

**Talento Altamente Especializado — Inteligencia Artificial 2026 — COCYTEN-Nayarit**

Instructor: M.Sc. Mario Iván López Valdovinos

Este notebook implementa y evalúa dos modelos YOLO (Ultralytics) sobre un **subconjunto de COCO (2017)**:
un modelo de **detección** (bounding boxes) y un modelo de **segmentación de instancias** (máscaras).

Estructura (sigue los 12 puntos del laboratorio):

1. Instalación de librerías
2. Descarga de COCO completo
3. Partición 70% train / 15% val / 15% test
4. Preparación de anotaciones (detección y segmentación)
5-6. Modelo de detección: carga y entrenamiento
7-8. Modelo de segmentación: carga y entrenamiento
9. Evaluación de ambos modelos en el conjunto de test
10. Métricas de detección
11. Métricas de segmentación
12. Visualización de resultados en imágenes de test

**Nota sobre la escala:** el enunciado del laboratorio permite explícitamente usar un subconjunto de
COCO ("Download the COCO dataset or use a smaller subset"). Esta versión usa un subconjunto de
`SUBSET_SIZE` imágenes (por defecto 2,500), descargadas **individualmente** (no como el .zip completo
de 5,000 imágenes de `val2017`, ~1 GB) — así solo se baja lo que realmente se va a usar, en vez del
dataset completo (~19 GB, 123,287 imágenes). Pensado para poder entrenar en una CPU sin GPU, en horas
en vez de días. Todo lo ajustable está centralizado en la Sección 0 (`SUBSET_SIZE`, `EPOCHS`,
`BATCH_SIZE`, `IMG_SIZE`).

**Nota sobre el split:** COCO2017 oficial trae `train2017` (imágenes+anotaciones), `val2017`
(imágenes+anotaciones) y `test2017` (solo imágenes, sin anotaciones públicas — se usan para el
servidor de evaluación oficial de COCO, no están disponibles para nosotros). Tomamos el subconjunto de
`val2017` (que ya viene anotado) y lo volvemos a partir nosotros mismos en 70/15/15.


## 1. Instalación de librerías

Python, PyTorch, Ultralytics YOLO, Matplotlib y `pycocotools` (para leer las anotaciones oficiales de COCO).

In [16]:
# Ejecuta esta celda una sola vez en tu entorno de VS Code / venv del curso.
# NOTA: si ya tienes torch instalado con soporte CUDA, no lo reinstales aquí (puede romper la versión con GPU).
%pip install -q ultralytics matplotlib pycocotools pyyaml opencv-python


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import os
import json
import random
import shutil
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
import torch
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
from ultralytics import YOLO

print("torch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())


torch: 2.13.0+cpu | CUDA disponible: False


## 0. Configuración central

Todos los valores que probablemente quieras ajustar según tu hardware están aquí, en un solo lugar.

In [18]:
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Rutas ---
COCO_RAW_ROOT = Path("./coco_raw").resolve()        # aquí se descarga/descomprime COCO2017 original
DATASET_ROOT = Path("./coco_lab_dataset").resolve()  # aquí se construye nuestro split 70/15/15

# --- Subconjunto y split ---
# El propio enunciado permite explícitamente usar un subconjunto ("Download the COCO
# dataset or use a smaller subset"), así que en vez de las 123,287 imágenes de
# train2017+val2017 combinadas, tomamos solo SUBSET_SIZE imágenes (de val2017, que ya
# viene anotado y pesa ~1 GB en vez de los ~18 GB de train2017) para que el
# entrenamiento sea manejable en una CPU sin GPU.
SUBSET_SIZE = 2500
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15  # el resto

# --- Entrenamiento ---
# CONFIGURACIÓN PARA CPU CON SUBCONJUNTO (SUBSET_SIZE imágenes, ver arriba).
# Con ~1,750 imágenes de entrenamiento (70% de 2,500) en vez de 86,300, cada época es
# mucho más rápida, así que podemos permitirnos más épocas y un tamaño de imagen más
# grande que en el escenario de dataset completo.
DET_WEIGHTS = "yolo11n.pt"       # modelo de detección preentrenado (COCO)
SEG_WEIGHTS = "yolo11n-seg.pt"   # modelo de segmentación preentrenado (COCO)
# yolo11n.pt / yolo11n-seg.pt ya vienen preentrenados en COCO completo (las mismas 80
# clases) -- no se entrena desde cero, solo se afina (fine-tune). Por eso pocas épocas
# ya son suficientes para cumplir el punto 6/8 del lab ("entrenar el modelo").
EPOCHS = 5               # fine-tuning desde pesos ya preentrenados en estas mismas clases
IMG_SIZE = 416           # compromiso entre velocidad y calidad de detección
BATCH_SIZE = 16          # baja esto (p. ej. 8 o 4) si te quedas sin memoria RAM
PATIENCE = 3             # detiene el entrenamiento si no mejora en 3 épocas seguidas (evita gastar tiempo de más)
CACHE_MODE = "disk"      # cachea imágenes preprocesadas en disco tras la 1a época (evita redecodificar JPEGs)
WORKERS = max(1, (os.cpu_count() or 4) - 1)  # hilos de carga de datos en paralelo, según tu CPU
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# --- Visualización ---
N_VIS_EXAMPLES = 6

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
COCO_RAW_ROOT.mkdir(parents=True, exist_ok=True)
print(f"COCO_RAW_ROOT = {COCO_RAW_ROOT}")
print(f"DATASET_ROOT  = {DATASET_ROOT}")
print(f"CPU detectado: {os.cpu_count()} núcleos -> WORKERS = {WORKERS}")


COCO_RAW_ROOT = E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_raw
DATASET_ROOT  = E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_dataset
CPU detectado: 6 núcleos -> WORKERS = 5


## 2. Descarga de las anotaciones de COCO

Aquí solo descargamos las anotaciones oficiales (~250 MB) — necesarias para saber qué imágenes
existen y elegir el subconjunto (Sección 3). Las imágenes en sí (`val2017`) se descargan más
adelante, una por una, **solo las `SUBSET_SIZE` que realmente se van a usar**, en vez de bajar el
.zip completo de 5,000 imágenes (~1 GB) para usar solo una fracción. La celda se puede volver a
ejecutar sin problema si se interrumpe, ya que no vuelve a descargar/descomprimir lo que ya existe.

In [19]:
# Descargamos SOLO las anotaciones aquí (~250 MB). Las imágenes NO se descargan como
# .zip completo -- más adelante (después de elegir el subconjunto) se descarga
# individualmente solo cada imagen que realmente vamos a usar, para no bajar de más.
COCO_FILES = {
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}


def download_with_progress(url, dest):
    def _hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(100, downloaded * 100 / total_size)
            print(f"\r  {dest.name}: {pct:5.1f}%", end="")
    urllib.request.urlretrieve(url, dest, reporthook=_hook)
    print()


for fname, url in COCO_FILES.items():
    dest = COCO_RAW_ROOT / fname
    if dest.exists():
        print(f"{fname} ya descargado, se omite.")
        continue
    print(f"Descargando {fname} ...")
    download_with_progress(url, dest)

for fname in COCO_FILES:
    marker = COCO_RAW_ROOT / (fname + ".extracted")
    if marker.exists():
        print(f"{fname} ya descomprimido, se omite.")
        continue
    print(f"Descomprimiendo {fname} ...")
    with zipfile.ZipFile(COCO_RAW_ROOT / fname) as zf:
        zf.extractall(COCO_RAW_ROOT)
    marker.touch()

print("Anotaciones descargadas y extraídas.")
print("Contenido de COCO_RAW_ROOT:", [p.name for p in COCO_RAW_ROOT.iterdir()])


annotations_trainval2017.zip ya descargado, se omite.
annotations_trainval2017.zip ya descomprimido, se omite.
Anotaciones descargadas y extraídas.
Contenido de COCO_RAW_ROOT: ['annotations', 'annotations_trainval2017.zip', 'annotations_trainval2017.zip.extracted', 'val2017']


## 3. Partición del dataset: 70% train / 15% val / 15% test

`test2017` de COCO no trae anotaciones públicas (se usa para el servidor de evaluación oficial). De
las 5,000 imágenes anotadas de `val2017`, tomamos aleatoriamente un subconjunto de `SUBSET_SIZE`
(2,500 por defecto) y lo partimos nosotros mismos en 70/15/15, a nivel de imagen completa (todas las
anotaciones de una imagen quedan en el mismo split).

In [20]:
ann_val_path = COCO_RAW_ROOT / "annotations" / "instances_val2017.json"

print("Cargando anotaciones (puede tardar un momento)...")
coco_val = COCO(str(ann_val_path))

# Lista completa de imágenes anotadas disponibles en val2017 (5,000 imágenes)
all_images = []
for image_id, image_info in coco_val.imgs.items():
    all_images.append({
        "coco_api": coco_val,
        "subdir": "val2017",
        "image_id": image_id,
        "file_name": image_info["file_name"],
        "width": image_info["width"],
        "height": image_info["height"],
    })

random.shuffle(all_images)

# Tomamos solo un subconjunto de SUBSET_SIZE imágenes (ver Sección 0) para que el
# entrenamiento sea manejable en CPU -- el enunciado permite explícitamente esto.
subset_images = all_images[:min(SUBSET_SIZE, len(all_images))]

n_total = len(subset_images)
n_train = int(TRAIN_FRACTION * n_total)
n_val = int(VAL_FRACTION * n_total)
# El resto va a test, para no perder/duplicar imágenes por redondeo
n_test = n_total - n_train - n_val

split_entries = {
    "train": subset_images[:n_train],
    "val": subset_images[n_train:n_train + n_val],
    "test": subset_images[n_train + n_val:],
}

print(f"Imágenes anotadas disponibles en val2017: {len(all_images)}")
print(f"Subconjunto usado para este laboratorio:  {n_total}")
for split_name, entries in split_entries.items():
    print(f"  {split_name:5s}: {len(entries):7d} imágenes ({100*len(entries)/n_total:.1f}%)")


Cargando anotaciones (puede tardar un momento)...
loading annotations into memory...
Done (t=0.87s)
creating index...
index created!
Imágenes anotadas disponibles en val2017: 5000
Subconjunto usado para este laboratorio:  2500
  train:    1750 imágenes (70.0%)
  val  :     375 imágenes (15.0%)
  test :     375 imágenes (15.0%)


### Descarga de solo las imágenes del subconjunto elegido

En vez de descargar las 5,000 imágenes de `val2017` para usar solo `SUBSET_SIZE`, descargamos
individualmente **únicamente** las imágenes que quedaron en `split_entries` (train + val + test),
usando la URL directa de cada una (`coco_url`, ya incluida en las anotaciones). Es reanudable: si se
interrumpe, al volver a correr esta celda salta las imágenes que ya existen.

In [21]:
val2017_dir = COCO_RAW_ROOT / "val2017"
val2017_dir.mkdir(parents=True, exist_ok=True)

# Unimos las entradas de los 3 splits -- son las únicas imágenes que de verdad necesitamos.
images_needed = [entry for entries in split_entries.values() for entry in entries]

downloaded, skipped, failed = 0, 0, 0
for idx, entry in enumerate(images_needed, start=1):
    dest = val2017_dir / entry["file_name"]
    if dest.exists():
        skipped += 1
        continue
    image_info = entry["coco_api"].imgs[entry["image_id"]]
    url = image_info.get("coco_url") or f"http://images.cocodataset.org/val2017/{entry['file_name']}"
    try:
        urllib.request.urlretrieve(url, dest)
        downloaded += 1
    except Exception as e:
        failed += 1
        print(f"  Falló {entry['file_name']}: {e}")

    if idx % 100 == 0 or idx == len(images_needed):
        print(f"\rProgreso: {idx}/{len(images_needed)} (nuevas: {downloaded}, ya existían: {skipped}, fallidas: {failed})", end="")

print()
print(f"Descarga del subconjunto completa: {downloaded} nuevas, {skipped} ya existían, {failed} fallidas.")
if failed:
    print("Vuelve a correr esta misma celda para reintentar las que fallaron (por timeouts, etc.).")



Descarga del subconjunto completa: 0 nuevas, 2500 ya existían, 0 fallidas.


## 4. Preparación de anotaciones

COCO guarda, para cada instancia, tanto la caja (`bbox`) como el polígono de segmentación
(`segmentation`) en la misma anotación — así que las convertimos a formato YOLO en un solo recorrido:

- **Detección:** `clase x_centro y_centro ancho alto` (normalizado 0-1)
- **Segmentación:** `clase x1 y1 x2 y2 ... xn yn` (polígono normalizado 0-1)

Las categorías de COCO usan IDs no contiguos (1-90, con huecos) — los remapeamos a índices
contiguos 0-79 para YOLO.

In [22]:
cat_ids = sorted(coco_val.getCatIds())
catid_to_yoloidx = {cat_id: idx for idx, cat_id in enumerate(cat_ids)}
class_names = [coco_val.loadCats(cat_id)[0]["name"] for cat_id in cat_ids]
NUM_CLASSES = len(class_names)
print(f"Número de clases COCO: {NUM_CLASSES}")
print("Primeras 5 clases:", class_names[:5])


Número de clases COCO: 80
Primeras 5 clases: ['person', 'bicycle', 'car', 'motorcycle', 'airplane']


In [23]:
def coco_ann_to_yolo_lines(coco_api, image_id, img_width, img_height):
    """Devuelve (lineas_deteccion, lineas_segmentacion) en formato YOLO para una imagen."""
    ann_ids = coco_api.getAnnIds(imgIds=image_id, iscrowd=False)
    anns = coco_api.loadAnns(ann_ids)

    det_lines, seg_lines = [], []
    for ann in anns:
        x, y, w, h = ann["bbox"]  # esquina superior-izq. + ancho/alto, en píxeles
        if w <= 0 or h <= 0:
            continue
        cls_idx = catid_to_yoloidx[ann["category_id"]]

        x_center_n = (x + w / 2) / img_width
        y_center_n = (y + h / 2) / img_height
        w_n = w / img_width
        h_n = h / img_height
        det_lines.append(f"{cls_idx} {x_center_n:.6f} {y_center_n:.6f} {w_n:.6f} {h_n:.6f}")

        segmentation = ann.get("segmentation")
        # Las anotaciones tipo RLE (objetos "crowd") no traen polígono utilizable; se omiten de seg.
        if isinstance(segmentation, list) and len(segmentation) > 0:
            polygon = max(segmentation, key=len)  # si hay varias partes, usamos la más grande
            if len(polygon) >= 6:  # al menos 3 puntos (x, y)
                coords = []
                for i in range(0, len(polygon) - 1, 2):
                    px = min(max(polygon[i] / img_width, 0.0), 1.0)
                    py = min(max(polygon[i + 1] / img_height, 0.0), 1.0)
                    coords.append(f"{px:.6f} {py:.6f}")
                seg_lines.append(f"{cls_idx} " + " ".join(coords))

    return det_lines, seg_lines


In [24]:
DET_VIEW = DATASET_ROOT / "det"
SEG_VIEW = DATASET_ROOT / "seg"


def link_or_copy(src: Path, dst: Path):
    """Symlink (rápido, no duplica espacio); si el sistema no lo permite (p. ej. Windows sin
    modo desarrollador), cae en copia real. El check es por ARCHIVO, nunca por carpeta --
    así un rerun parcial siempre puede completar lo que faltó, sin quedar con copias
    "congeladas" desactualizadas."""
    if dst.exists() or dst.is_symlink():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def materialize_split(split_name, entries):
    """Escribe imágenes (symlink/copia) + labels de detección y de segmentación DIRECTAMENTE
    en las carpetas det/ y seg/ (sin una capa intermedia de symlink de carpeta completa, que
    en Windows sin modo desarrollador degenera en una copia estática que nunca se actualiza)."""
    det_images_out = DET_VIEW / "images" / split_name
    det_labels_out = DET_VIEW / "labels" / split_name
    seg_images_out = SEG_VIEW / "images" / split_name
    seg_labels_out = SEG_VIEW / "labels" / split_name
    for d in (det_images_out, det_labels_out, seg_images_out, seg_labels_out):
        d.mkdir(parents=True, exist_ok=True)

    for entry in entries:
        src_img = COCO_RAW_ROOT / entry["subdir"] / entry["file_name"]
        link_or_copy(src_img, det_images_out / entry["file_name"])
        link_or_copy(src_img, seg_images_out / entry["file_name"])

        det_lines, seg_lines = coco_ann_to_yolo_lines(
            entry["coco_api"], entry["image_id"], entry["width"], entry["height"]
        )
        stem = Path(entry["file_name"]).stem
        (det_labels_out / f"{stem}.txt").write_text("\n".join(det_lines))
        (seg_labels_out / f"{stem}.txt").write_text("\n".join(seg_lines))


for split_name, entries in split_entries.items():
    print(f"Materializando split '{split_name}' ({len(entries)} imágenes)...")
    materialize_split(split_name, entries)

print("Anotaciones de detección y segmentación listas (en det/ y seg/).")


Materializando split 'train' (1750 imágenes)...
Materializando split 'val' (375 imágenes)...
Materializando split 'test' (375 imágenes)...
Anotaciones de detección y segmentación listas (en det/ y seg/).


Ultralytics espera que, para cada carpeta `images/<split>`, exista una carpeta hermana
`labels/<split>` (reemplazando `images` por `labels` en la ruta). Como necesitamos **dos** conjuntos de
labels distintos (detección y segmentación) para las **mismas** imágenes, escribimos las imágenes
(por symlink o copia, según lo permita el sistema operativo) directamente en dos "vistas" del dataset
(`det/` y `seg/`), archivo por archivo -- así, si el notebook se interrumpe y se vuelve a correr, cada
imagen se completa de forma independiente sin dejar copias parciales desactualizadas.

In [25]:
# Ya no hace falta nada aquí: la Sección 4 ya escribió las imágenes y etiquetas
# directamente en DET_VIEW y SEG_VIEW. Solo confirmamos que quedaron completas.
for view_name, view_root in [("det", DET_VIEW), ("seg", SEG_VIEW)]:
    for split_name in split_entries:
        n_images = len(list((view_root / "images" / split_name).glob("*")))
        n_labels = len(list((view_root / "labels" / split_name).glob("*.txt")))
        print(f"{view_name}/{split_name}: {n_images} imágenes, {n_labels} archivos de etiquetas")


det/train: 3500 imágenes, 1750 archivos de etiquetas
det/val: 750 imágenes, 375 archivos de etiquetas
det/test: 375 imágenes, 375 archivos de etiquetas
seg/train: 3500 imágenes, 1750 archivos de etiquetas
seg/val: 750 imágenes, 375 archivos de etiquetas
seg/test: 375 imágenes, 375 archivos de etiquetas


In [26]:
def write_dataset_yaml(path, dataset_view_root):
    config = {
        "path": str(dataset_view_root),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {idx: name for idx, name in enumerate(class_names)},
    }
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)
    return path


det_yaml_path = write_dataset_yaml(DATASET_ROOT / "coco_lab_det.yaml", DET_VIEW)
seg_yaml_path = write_dataset_yaml(DATASET_ROOT / "coco_lab_seg.yaml", SEG_VIEW)

print("YAML de detección:  ", det_yaml_path)
print("YAML de segmentación:", seg_yaml_path)


YAML de detección:   E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_dataset\coco_lab_det.yaml
YAML de segmentación: E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_dataset\coco_lab_seg.yaml


## 5-6. Modelo de detección: carga y entrenamiento

Cargamos `yolo11n.pt` (preentrenado en COCO) y lo afinamos (`fine-tune`) sobre nuestro propio split
70/15/15.

**Cómo pausar y reanudar (importante si no puedes dejar la PC prendida varios días):**

- Para pausar: interrumpe el kernel de Jupyter (botón de "stop" / `Ctrl+C` en terminal), **no apagues la
  PC de golpe a mitad de una escritura de checkpoint** — espera a que termine la época en curso
  (Ultralytics guarda `last.pt` al final de cada época, no a mitad de una).
- Para reanudar: simplemente vuelve a correr esta misma celda más tarde. Detecta automáticamente si ya
  existe `coco_lab_runs/<detection|segmentation>/weights/last.pt` y continúa desde ahí en vez de
  reiniciar desde el modelo preentrenado.
- Si quieres empezar de cero (por ejemplo, después de cambiar `EPOCHS` o `IMG_SIZE`), borra la carpeta
  `coco_lab_runs/detection` o `coco_lab_runs/segmentation` correspondiente antes de volver a correr.


In [27]:
# 5. Cargar modelo de detección preentrenado.
# Si ya existe un entrenamiento previo interrumpido (last.pt), se reanuda desde ahí en
# vez de empezar de cero -- así puedes apagar la PC entre sesiones sin perder progreso.
#det_last_ckpt = Path("coco_lab_runs/detection/weights/last.pt")
det_last_ckpt = Path("runs/detect/coco_lab_runs/detection/weights/last.pt")

if det_last_ckpt.exists():
    print(f"Reanudando entrenamiento de detección desde: {det_last_ckpt}")
    det_model = YOLO(str(det_last_ckpt))
    det_train_results = det_model.train(resume=True)
else:
    print("Iniciando entrenamiento de detección desde cero.")
    det_model = YOLO(DET_WEIGHTS)
    det_train_results = det_model.train(   # 6. Entrenar con el split de entrenamiento
        data=str(det_yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
        project="coco_lab_runs",
        name="detection",
        val=True,        # valida contra el split "val" al final de cada época
        patience=PATIENCE,
        cache=CACHE_MODE,
        workers=WORKERS,
    )


Reanudando entrenamiento de detección desde: runs\detect\coco_lab_runs\detection\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.114 available  Update with 'pip install -U ultralytics'
WARNING model 'runs\detect\coco_lab_runs\detection\weights\last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.106  Python-3.13.5 torch-2.13.0+cpu CPU (13th Gen Intel Core i3-1305U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False

## 7-8. Modelo de segmentación: carga y entrenamiento

Cargamos `yolo11n-seg.pt` (preentrenado en COCO con cabeza de segmentación) y lo afinamos sobre la
misma partición de datos, usando las etiquetas de polígono en vez de solo cajas.

**Cómo pausar y reanudar (importante si no puedes dejar la PC prendida varios días):**

- Para pausar: interrumpe el kernel de Jupyter (botón de "stop" / `Ctrl+C` en terminal), **no apagues la
  PC de golpe a mitad de una escritura de checkpoint** — espera a que termine la época en curso
  (Ultralytics guarda `last.pt` al final de cada época, no a mitad de una).
- Para reanudar: simplemente vuelve a correr esta misma celda más tarde. Detecta automáticamente si ya
  existe `coco_lab_runs/<detection|segmentation>/weights/last.pt` y continúa desde ahí en vez de
  reiniciar desde el modelo preentrenado.
- Si quieres empezar de cero (por ejemplo, después de cambiar `EPOCHS` o `IMG_SIZE`), borra la carpeta
  `coco_lab_runs/detection` o `coco_lab_runs/segmentation` correspondiente antes de volver a correr.


In [28]:
# 7. Cargar modelo de segmentación preentrenado (mismo mecanismo de reanudación que detección).
#seg_last_ckpt = Path("coco_lab_runs/segmentation/weights/last.pt")
seg_last_ckpt = Path("runs/segment/coco_lab_runs/segmentation/weights/last.pt")

if seg_last_ckpt.exists():
    print(f"Reanudando entrenamiento de segmentación desde: {seg_last_ckpt}")
    seg_model = YOLO(str(seg_last_ckpt))
    seg_train_results = seg_model.train(resume=True)
else:
    print("Iniciando entrenamiento de segmentación desde cero.")
    seg_model = YOLO(SEG_WEIGHTS)
    seg_train_results = seg_model.train(   # 8. Entrenar con el split de entrenamiento
        data=str(seg_yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        seed=SEED,
        project="coco_lab_runs",
        name="segmentation",
        val=True,
        patience=PATIENCE,
        cache=CACHE_MODE,
        workers=WORKERS,
    )


Reanudando entrenamiento de segmentación desde: runs\segment\coco_lab_runs\segmentation\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.114 available  Update with 'pip install -U ultralytics'
WARNING model 'runs\segment\coco_lab_runs\segmentation\weights\last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.106  Python-3.13.5 torch-2.13.0+cpu CPU (13th Gen Intel Core i3-1305U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8-seg.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0

## 9. Evaluación de ambos modelos en el conjunto de test

Ultralytics separa claramente `val` (usado durante el entrenamiento) de `test` (reservado hasta el
final). Aquí evaluamos explícitamente contra `split="test"`, que el modelo nunca vio durante el
entrenamiento.

In [29]:
det_test_metrics = det_model.val(
    data=str(det_yaml_path),
    split="test",
    imgsz=IMG_SIZE,
    device=DEVICE,
)


Ultralytics 8.4.106  Python-3.13.5 torch-2.13.0+cpu CPU (13th Gen Intel Core i3-1305U)
YOLO11n summary (fused): 101 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 16.37.2 MB/s, size: 236.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_dataset\det\labels\test.cache... 375 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 375/375 65.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 1.5s/it 36.8s1.6ss
                   all        375       2623      0.619      0.456      0.507      0.356
                person        194        805       0.74      0.576       0.65      0.432
               bicycle         10         19        

In [30]:
seg_test_metrics = seg_model.val(
    data=str(seg_yaml_path),
    split="test",
    imgsz=IMG_SIZE,
    device=DEVICE,
)


Ultralytics 8.4.106  Python-3.13.5 torch-2.13.0+cpu CPU (13th Gen Intel Core i3-1305U)
YOLO11n-seg summary (fused): 114 layers, 2,868,664 parameters, 0 gradients, 9.7 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 10.75.7 MB/s, size: 142.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_dataset\seg\labels\test.cache... 375 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 375/375 68.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.3s/it 55.2s2.2ss
                   all        375       2623      0.588      0.458      0.495      0.346      0.606      0.423       0.48      0.303
                person        194        805     

## 10. Métricas de detección (conjunto de test)

Precisión, recall, F1-score, mAP@0.50 y mAP@0.50:0.95. Ultralytics reporta precisión/recall/mAP
directamente; el F1 lo derivamos de precisión y recall medias.

In [31]:
det_precision = float(det_test_metrics.box.mp)
det_recall = float(det_test_metrics.box.mr)
det_f1 = 2 * det_precision * det_recall / (det_precision + det_recall + 1e-9)
det_map50 = float(det_test_metrics.box.map50)
det_map5095 = float(det_test_metrics.box.map)

print("Métricas de detección — conjunto de test")
print(f"  Precisión (media sobre clases): {det_precision:.4f}")
print(f"  Recall (media sobre clases):    {det_recall:.4f}")
print(f"  F1-score:                       {det_f1:.4f}")
print(f"  mAP@0.50:                       {det_map50:.4f}")
print(f"  mAP@0.50:0.95:                  {det_map5095:.4f}")


Métricas de detección — conjunto de test
  Precisión (media sobre clases): 0.6194
  Recall (media sobre clases):    0.4557
  F1-score:                       0.5251
  mAP@0.50:                       0.5075
  mAP@0.50:0.95:                  0.3563


## 11. Métricas de segmentación (conjunto de test)

Mask precision, mask recall, mask mAP@0.50 y mAP@0.50:0.95 los reporta Ultralytics directamente
(están basados en IoU de máscara para decidir aciertos, igual que mAP de detección usa IoU de caja).

Adicionalmente, el laboratorio pide **IoU** y **coeficiente Dice** explícitos. Los calculamos a nivel
de píxel sobre una máscara combinada (unión de todas las instancias) por imagen, comparando la máscara
predicha contra la máscara real, acumulando sobre el conjunto de test (con el subconjunto de
`SUBSET_SIZE` imágenes, el test set ya es pequeño -- unas ~375 imágenes con la configuración por
defecto -- así que se puede evaluar completo sin que sea lento).

In [32]:
seg_mask_precision = float(seg_test_metrics.seg.mp)
seg_mask_recall = float(seg_test_metrics.seg.mr)
seg_mask_map50 = float(seg_test_metrics.seg.map50)
seg_mask_map5095 = float(seg_test_metrics.seg.map)

print("Métricas de segmentación — conjunto de test (Ultralytics)")
print(f"  Precisión de máscara (media): {seg_mask_precision:.4f}")
print(f"  Recall de máscara (media):    {seg_mask_recall:.4f}")
print(f"  Mask mAP@0.50:                {seg_mask_map50:.4f}")
print(f"  Mask mAP@0.50:0.95:           {seg_mask_map5095:.4f}")


Métricas de segmentación — conjunto de test (Ultralytics)
  Precisión de máscara (media): 0.6056
  Recall de máscara (media):    0.4234
  Mask mAP@0.50:                0.4796
  Mask mAP@0.50:0.95:           0.3027


In [33]:
import cv2

N_IOU_SAMPLE = len(split_entries["test"])  # con el subconjunto chico, evaluamos el test set completo


def rasterize_gt_mask(coco_api, image_id, width, height):
    """Combina (unión) todas las máscaras de instancia de una imagen en una sola máscara binaria."""
    ann_ids = coco_api.getAnnIds(imgIds=image_id, iscrowd=False)
    anns = coco_api.loadAnns(ann_ids)
    combined = np.zeros((height, width), dtype=np.uint8)
    for ann in anns:
        segmentation = ann.get("segmentation")
        if isinstance(segmentation, list) and len(segmentation) > 0:
            rles = mask_utils.frPyObjects(segmentation, height, width)
            rle = mask_utils.merge(rles)
            m = mask_utils.decode(rle)
        elif isinstance(segmentation, dict):
            m = mask_utils.decode(segmentation)
        else:
            continue
        combined = np.maximum(combined, m.astype(np.uint8))
    return combined


test_entries_sample = split_entries["test"][:N_IOU_SAMPLE]

total_intersection = 0
total_union = 0
total_gt_area = 0
total_pred_area = 0

for entry in test_entries_sample:
    img_path = SEG_VIEW / "images" / "test" / entry["file_name"]
    gt_mask = rasterize_gt_mask(entry["coco_api"], entry["image_id"], entry["width"], entry["height"])

    prediction = seg_model.predict(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    if prediction.masks is not None:
        pred_mask = prediction.masks.data.cpu().numpy().max(axis=0)
        pred_mask = (pred_mask > 0.5).astype(np.uint8)
        if pred_mask.shape != gt_mask.shape:
            pred_mask = cv2.resize(
                pred_mask, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST
            )
    else:
        pred_mask = np.zeros_like(gt_mask)

    total_intersection += int(np.logical_and(gt_mask, pred_mask).sum())
    total_union += int(np.logical_or(gt_mask, pred_mask).sum())
    total_gt_area += int(gt_mask.sum())
    total_pred_area += int(pred_mask.sum())

dataset_iou = total_intersection / total_union if total_union > 0 else float("nan")
denom = total_gt_area + total_pred_area
dataset_dice = (2 * total_intersection / denom) if denom > 0 else float("nan")

print(f"IoU a nivel de píxel (máscara combinada, {len(test_entries_sample)} imágenes de test):   {dataset_iou:.4f}")
print(f"Dice a nivel de píxel (mismas {len(test_entries_sample)} imágenes):                       {dataset_dice:.4f}")
print("\nNota: Dice = 2*IoU / (1+IoU) matemáticamente, así que ambos números son consistentes entre sí.")


IoU a nivel de píxel (máscara combinada, 375 imágenes de test):   0.7103
Dice a nivel de píxel (mismas 375 imágenes):                       0.8306

Nota: Dice = 2*IoU / (1+IoU) matemáticamente, así que ambos números son consistentes entre sí.


## 12. Visualización de resultados en imágenes de test

Para varias imágenes de test se muestran, lado a lado: **ground-truth** (cajas + máscaras reales) y
**predicción** (cajas + máscaras predichas + puntuación de confianza) del modelo de segmentación.

In [34]:
import matplotlib.image as mpimg


def draw_ground_truth(ax, entry):
    img_path = SEG_VIEW / "images" / "test" / entry["file_name"]
    image = mpimg.imread(img_path)
    ax.imshow(image)

    ann_ids = entry["coco_api"].getAnnIds(imgIds=entry["image_id"], iscrowd=False)
    anns = entry["coco_api"].loadAnns(ann_ids)

    for ann in anns:
        x, y, w, h = ann["bbox"]
        cls_name = class_names[catid_to_yoloidx[ann["category_id"]]]
        rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x, max(y - 4, 0), cls_name, color="lime", fontsize=8, backgroundcolor="black")

        segmentation = ann.get("segmentation")
        if isinstance(segmentation, list):
            for polygon in segmentation:
                if len(polygon) >= 6:
                    pts = np.array(polygon).reshape(-1, 2)
                    ax.add_patch(patches.Polygon(pts, closed=True, facecolor="lime", alpha=0.25, edgecolor=None))

    ax.set_title("Ground truth")
    ax.axis("off")


def draw_prediction(ax, entry):
    img_path = SEG_VIEW / "images" / "test" / entry["file_name"]
    result = seg_model.predict(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    # result.plot() dibuja cajas, máscaras, nombres de clase y confianza automáticamente
    plotted_bgr = result.plot(boxes=True, masks=True, labels=True, conf=True)
    ax.imshow(plotted_bgr[..., ::-1])  # BGR (OpenCV) -> RGB (matplotlib)
    ax.set_title("Predicción (caja + máscara + confianza)")
    ax.axis("off")


VIS_OUTPUT_DIR = Path("./coco_lab_visualizations").resolve()
VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vis_entries = split_entries["test"][:N_VIS_EXAMPLES]

for entry in vis_entries:
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    draw_ground_truth(axes[0], entry)
    draw_prediction(axes[1], entry)
    fig.suptitle(entry["file_name"])
    plt.tight_layout()

    stem = Path(entry["file_name"]).stem
    save_path = VIS_OUTPUT_DIR / f"{stem}_gt_vs_pred.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()
    plt.close(fig)

print(f"Imágenes guardadas en: {VIS_OUTPUT_DIR}")

<Figure size 1400x700 with 2 Axes>

<Figure size 1400x700 with 2 Axes>

<Figure size 1400x700 with 2 Axes>

<Figure size 1400x700 with 2 Axes>

<Figure size 1400x700 with 2 Axes>

<Figure size 1400x700 with 2 Axes>

Imágenes guardadas en: E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_visualizations


In [35]:
import matplotlib.image as mpimg  # para leer la imagen original como arreglo de píxeles

from pathlib import Path
det_model = YOLO("runs/detect/coco_lab_runs/detection/weights/best.pt")
seg_model = YOLO("runs/segment/coco_lab_runs/segmentation/weights/best.pt")

def draw_original(ax, entry):
    img_path = SEG_VIEW / "images" / "test" / entry["file_name"]
    image = mpimg.imread(img_path)  # imagen tal cual, sin ninguna anotación
    ax.imshow(image)
    ax.set_title("Imagen original")
    ax.axis("off")


def draw_detection_prediction(ax, entry):
    img_path = DET_VIEW / "images" / "test" / entry["file_name"]
    result = det_model.predict(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]  # modelo de detección
    plotted_bgr = result.plot(boxes=True, labels=True, conf=True)  # solo cajas, sin máscaras
    ax.imshow(plotted_bgr[..., ::-1])  # BGR (OpenCV) -> RGB (matplotlib)
    ax.set_title("Predicción de detección")
    ax.axis("off")


def draw_segmentation_prediction(ax, entry):
    img_path = SEG_VIEW / "images" / "test" / entry["file_name"]
    result = seg_model.predict(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]  # modelo de segmentación
    plotted_bgr = result.plot(boxes=False, masks=True, labels=True, conf=True)  # solo máscaras, sin cajas
    ax.imshow(plotted_bgr[..., ::-1])
    ax.set_title("Predicción de segmentación")
    ax.axis("off")


TRIPTYCH_OUTPUT_DIR = Path("./coco_lab_triptych").resolve()
TRIPTYCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

triptych_entries = split_entries["test"][:N_VIS_EXAMPLES]

for entry in triptych_entries:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # panel de 3: original | detección | segmentación
    draw_original(axes[0], entry)
    draw_detection_prediction(axes[1], entry)
    draw_segmentation_prediction(axes[2], entry)
    fig.suptitle(entry["file_name"])
    plt.tight_layout()

    stem = Path(entry["file_name"]).stem
    save_path = TRIPTYCH_OUTPUT_DIR / f"{stem}_original_det_seg.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")  # guarda la figura como PNG

    plt.show()
    plt.close(fig)

print(f"Imágenes guardadas en: {TRIPTYCH_OUTPUT_DIR}")

<Figure size 1800x600 with 3 Axes>

<Figure size 1800x600 with 3 Axes>

<Figure size 1800x600 with 3 Axes>

<Figure size 1800x600 with 3 Axes>

<Figure size 1800x600 with 3 Axes>

<Figure size 1800x600 with 3 Axes>

Imágenes guardadas en: E:\28\3. Taller Altamente Especializado IA - CINVESTAV\Módulo 4\Laboratorios\Segmentation_Detection\coco_lab_triptych


## Resumen final

- **Detección** (`yolo11n.pt` afinado): precisión, recall, F1, mAP@0.50 y mAP@0.50:0.95 — Sección 10.
- **Segmentación** (`yolo11n-seg.pt` afinado): precisión/recall de máscara, mask mAP@0.50 y
  mAP@0.50:0.95 (Ultralytics), más IoU y Dice a nivel de píxel calculados manualmente — Sección 11.
- Ejemplos visuales con ground-truth vs. predicción — Sección 12.

Los pesos entrenados quedan guardados automáticamente por Ultralytics en
`coco_lab_runs/detection/weights/best.pt` y `coco_lab_runs/segmentation/weights/best.pt`.

**Recuerda guardar este archivo exactamente como `COCO_lab.ipynb`**, según lo pide el enunciado.